# Clinical Data Auditor Simulation: Day 28

In [22]:
import pandas as pd
import sqlite3
# creating dataset tracking the number of telemetry packets received per minute per bed
telemetry_data = {
    "log_minute": ["14:01", "14:01", "14:01", "14:02", "14:02", "14:02", "14:02"],
    "bed_id": ["Bed-01", "Bed-02", "Bed-03", "Bed-01", "Bed-02", "Bed-02", "Bed-03"],
    "packet_id": ["T-101", "T-102", "T-103", "T-104", "T-105", "T-106", "T-107"]
}
# Let's simulate an automated flood anomaly where Bed-02 transmits 500 times in minute 14:03
flood_records = []
for i in range(500):
    flood_records.append({"log_minute": "14:03", "bed_id": "Bed-02", "packet_id": f"T-FLOOD-{i}"})

# Add normal traffic for minute 14:03
flood_records.append({"log_minute": "14:03", "bed_id": "Bed-01", "packet_id": "T-108"})
flood_records.append({"log_minute": "14:03", "bed_id": "Bed-03", "packet_id": "T-109"})

df_base = pd.DataFrame(telemetry_data)
df_flood = pd.DataFrame(flood_records)
df_telemetry = pd.concat([df_base, df_flood], ignore_index=True)

# 2. Connect to SQL
conn = sqlite3.connect(":memory:")
df_telemetry.to_sql("network_traffic", conn, index=False, if_exists="replace")

def run_query(query):
    return pd.read_sql_query(query, conn)
print("******************************* Telemetry Flood Audit Database is ready! **********************")

******************************* Telemetry Flood Audit Database is ready! **********************


# Profiling Volume Traffic

In [20]:
# SQL query for all datat to review
all_data = "SELECT * FROM network_traffic"
print("********************************* all data to review ******************")
display(run_query(all_data))
print()
# SQL query that counts the total number of packets received, grouped by log_minute and bed_id. Order the results by the packet count in descending order (DESC).
packet_count = """
SELECT
    log_minute,
    bed_id,
    COUNT(packet_id) AS packet_count
FROM network_traffic
GROUP BY log_minute, bed_id
ORDER BY packet_count DESC;
"""
print("********************************* packet count *******************")
display(run_query(packet_count))

********************************* all data to review ******************


,log_minute,bed_id,packet_id
0,14:01,Bed-01,T-101
1,14:01,Bed-02,T-102
2,14:01,Bed-03,T-103
3,14:02,Bed-01,T-104
4,14:02,Bed-02,T-105
5,14:02,Bed-02,T-106
6,14:02,Bed-03,T-107



********************************* packet count *******************


,log_minute,bed_id,packet_count
0,14:02,Bed-02,2
1,14:01,Bed-01,1
2,14:01,Bed-02,1
3,14:01,Bed-03,1
4,14:02,Bed-01,1
5,14:02,Bed-03,1


# Setting an Automated Alert Threshold

In [24]:
# SQL query to filter out normal traffic and display only the instances where a specific bed transmits more than 50 packets within a single minute.
flood_query = """
SELECT 
    log_minute,
    bed_id,
    COUNT(packet_id) AS total_packets_received
FROM network_traffic
GROUP BY log_minute, bed_id
HAVING COUNT(packet_id) > 50
"""
print("************************** flood query ************************")
result = run_query(flood_query)
print(result)
#display(run_query(normal_traffic))

************************** flood query ************************
  log_minute  bed_id  total_packets_received
0      14:03  Bed-02                     500
